# 13 — Origin Classification: Hierarchical Region → Country

Two-stage pipeline.

**Stage 1**: RoBERTa classifier predicts one of 4 regions (East Africa / Central America / South America / Asia-Pacific) from the scrubbed+ full-concat review text.

**Stage 2**: one RoBERTa country classifier per region, trained only on rows in that region. At inference, Stage 1 picks the region, and that region's Stage 2 head picks the country among the countries in its region.

Evaluation reports three numbers:
- **Stage 1 region F1** — how well the first classifier identifies the region.
- **Oracle-region country F1** — Stage 2 quality *given a perfect Stage 1*. This is the upper bound of the pipeline.
- **End-to-end country F1** — actual pipeline performance (cascade errors included).

Plus an error decomposition: of the country errors, how many came from wrong-region predictions vs wrong-country-within-correct-region.

- Input: `text_full_concat_scrubbed_plus` (4-tier scrubbed, full concat)
- Model: RoBERTa-base (weighted CE, lr=2e-5) for all stages
- Seeds: [42, 123, 2024]
- Training runs total: 3 (Stage 1) + 4 × 3 (Stage 2) = 15 fine-tunes

Reference — flat country-level result (07.1): RoBERTa 0.6427 ± 0.0068, 15 classes, 6,820 rows.
Reference — flat 4-way regional (11): RoBERTa 0.8171 ± 0.0071, 4 classes, 7,585 rows.


In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
MAX_LENGTH = 256
COUNTRY_FLOOR = 10  # drop countries with fewer rows than this

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
USE_FP16 = True

ROBERTA_CKPT = 'roberta-base'
ROBERTA_LR = 2e-5

OUTPUT_DIR_ROOT = 'artifacts/origin_hierarchical_region_to_country_scrubbed_plus'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

COUNTRY_TO_REGION = {
    # --- East Africa ---
    'Ethiopia': 'East Africa',
    'Kenya': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Tanzania': 'East Africa',
    'Uganda': 'East Africa',
    'DR Congo': 'East Africa',
    'Zambia': 'East Africa',
    'Zimbabwe': 'East Africa',
    'Cameroon': 'East Africa',
    'Malawi': 'East Africa',
    'South Africa': 'East Africa',
    'Yemen': 'East Africa',
    # --- Central America ---
    'Guatemala': 'Central America',
    'Costa Rica': 'Central America',
    'Panama': 'Central America',
    'El Salvador': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Mexico': 'Central America',
    'Jamaica': 'Central America',
    'Puerto Rico': 'Central America',
    'Haiti': 'Central America',
    'Dominican Republic': 'Central America',
    # --- South America ---
    'Colombia': 'South America',
    'Peru': 'South America',
    'Brazil': 'South America',
    'Ecuador': 'South America',
    'Bolivia': 'South America',
    'Venezuela': 'South America',
    # --- Asia-Pacific ---
    'Indonesia': 'Asia-Pacific',
    'Taiwan': 'Asia-Pacific',
    'Thailand': 'Asia-Pacific',
    'Papua New Guinea': 'Asia-Pacific',
    'Philippines': 'Asia-Pacific',
    'India': 'Asia-Pacific',
    'Vietnam': 'Asia-Pacific',
    'China': 'Asia-Pacific',
    'Timor-Leste': 'Asia-Pacific',
    'Malaysia': 'Asia-Pacific',
    'Laos': 'Asia-Pacific',
    'Nepal': 'Asia-Pacific',
    'Myanmar': 'Asia-Pacific',
    'Australia': 'Asia-Pacific',
    'United Kingdom': 'Asia-Pacific',
    'United States': 'Asia-Pacific',
}


Device: cuda


## Scrubbing vocabulary (identical to 07.1 / 04.5 / 11)


In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)


Loaded 403 scrub terms


## Build text, country, region columns; apply country floor


In [3]:
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)
df['origin_region'] = df['origin_country'].map(COUNTRY_TO_REGION)

work = df[
    (df['text_raw_minimal'].str.len() >= 30)
    & (df['origin_country'].notna())
    & (df['origin_region'].notna())
].copy().reset_index(drop=True)

counts = work['origin_country'].value_counts()
valid_countries = counts[counts >= COUNTRY_FLOOR].index
work = work[work['origin_country'].isin(valid_countries)].copy().reset_index(drop=True)

def contains_own_country(row):
    t = row['text_full_concat_scrubbed_plus'].lower()
    c = str(row['origin_country']).lower()
    return c in t if c and t else False
work['leaks_country'] = work.apply(contains_own_country, axis=1)
leak_rate = float(work['leaks_country'].mean())

print(f'Rows: {len(work)} | Countries: {work["origin_country"].nunique()} | Regions: {work["origin_region"].nunique()}')
print(f'Country-name leakage: {leak_rate:.2%}')
print()
print('Region distribution:')
print(work['origin_region'].value_counts())
print()
print('Country distribution:')
print(work['origin_country'].value_counts())


Rows: 7534 | Countries: 32 | Regions: 4
Country-name leakage: 0.89%

Region distribution:
origin_region
East Africa        3128
Central America    1936
South America      1444
Asia-Pacific       1026
Name: count, dtype: int64

Country distribution:
origin_country
Ethiopia              1996
Colombia               988
Kenya                  699
Guatemala              561
Indonesia              452
Costa Rica             373
Panama                 354
United States          302
El Salvador            240
Brazil                 200
Rwanda                 176
Peru                   130
Nicaragua              125
Honduras               123
Mexico                 101
Burundi                 85
Papua New Guinea        79
Ecuador                 75
Taiwan                  71
Tanzania                68
Thailand                68
Bolivia                 51
Yemen                   48
India                   38
DR Congo                34
Jamaica                 26
Philippines             16
Uganda 

## Top-level stratified split (shared across Stage 1 and Stage 2)


In [4]:
# Stratify by country (finer-grained than region) so each country has test support.
idx = np.arange(len(work))
y_country = work['origin_country'].tolist()

idx_tmp, idx_test = train_test_split(
    idx, test_size=0.15, stratify=y_country, random_state=RANDOM_STATE,
)
y_tmp_country = [y_country[i] for i in idx_tmp]
idx_train, idx_val = train_test_split(
    idx_tmp, test_size=0.15/0.85, stratify=y_tmp_country, random_state=RANDOM_STATE,
)

train_df = work.iloc[idx_train].copy().reset_index(drop=True)
val_df   = work.iloc[idx_val].copy().reset_index(drop=True)
test_df  = work.iloc[idx_test].copy().reset_index(drop=True)

region_names  = sorted(work['origin_region'].unique().tolist())
country_names = sorted(work['origin_country'].unique().tolist())
region2id  = {r: i for i, r in enumerate(region_names)}
country2id = {c: i for i, c in enumerate(country_names)}
id2region  = {i: r for r, i in region2id.items()}
id2country = {i: c for c, i in country2id.items()}

print(f'Train/Val/Test: {len(train_df)} / {len(val_df)} / {len(test_df)}')
print(f'Regions ({len(region_names)}): {region_names}')
print(f'Countries ({len(country_names)})')


Train/Val/Test: 5273 / 1130 / 1131
Regions (4): ['Asia-Pacific', 'Central America', 'East Africa', 'South America']
Countries (32)


## Dataset + run_one_with_preds helper


In [5]:
class CoffeeOriginDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def run_one_with_preds(
    tag, seed, model_checkpoint, learning_rate, use_class_weights,
    train_texts, train_labels, val_texts, val_labels,
    test_texts_for_preds, label_names,
    epochs=12, early_stop_patience=3,
):
    """Train and return full test-set predictions for later routing."""
    set_seed(seed)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    id2lab = {i: n for i, n in enumerate(label_names)}
    lab2id = {n: i for i, n in enumerate(label_names)}

    train_ds = CoffeeOriginDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_ds   = CoffeeOriginDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
    # Dummy labels for the "test-for-prediction" set — we only want predictions.
    pred_ds  = CoffeeOriginDataset(test_texts_for_preds, [0]*len(test_texts_for_preds), tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels=len(label_names), id2label=id2lab, label2id=lab2id,
    )

    args = TrainingArguments(
        output_dir=out_dir, learning_rate=learning_rate,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=epochs,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch', logging_strategy='epoch',
        save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro', greater_is_better=True,
        disable_tqdm=True, report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(), seed=seed,
    )
    trainer_cls = WeightedTrainer if use_class_weights else Trainer
    cw_tensor = None
    extra = {}
    if use_class_weights:
        cw = compute_class_weight(class_weight='balanced',
                                  classes=np.arange(len(label_names)),
                                  y=train_labels)
        cw_tensor = torch.tensor(cw, dtype=torch.float32)
        extra = dict(class_weights=cw_tensor)

    callbacks = [EarlyStoppingCallback(early_stopping_patience=early_stop_patience)]
    trainer = trainer_cls(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics, callbacks=callbacks, **extra,
    )
    try: trainer.remove_callback(NotebookProgressCallback)
    except Exception: pass
    print(f'\n=== {tag} | {model_checkpoint} | lr={learning_rate} | weighted={use_class_weights} | seed={seed} ===')
    trainer.train()
    val_m = trainer.evaluate(eval_dataset=val_ds)
    # Predictions on the test-for-prediction set
    pred_logits = trainer.predict(pred_ds).predictions
    pred_ids = np.argmax(pred_logits, axis=1)

    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'tag': tag, 'seed': seed,
        'val_f1_macro': val_m['eval_f1_macro'],
        'val_bal_acc': val_m['eval_balanced_accuracy'],
        'val_accuracy': val_m['eval_accuracy'],
        'test_pred_ids': pred_ids.tolist(),
        'label_names': list(label_names),
    }


## Stage 1 — 4-way region classifier × 3 seeds


In [6]:
# Prepare Stage 1 data (targets = origin_region)
s1_train_texts  = train_df[TEXT_COLUMN].tolist()
s1_val_texts    = val_df[TEXT_COLUMN].tolist()
s1_test_texts   = test_df[TEXT_COLUMN].tolist()
s1_train_labels = [region2id[r] for r in train_df['origin_region']]
s1_val_labels   = [region2id[r] for r in val_df['origin_region']]
s1_test_labels  = [region2id[r] for r in test_df['origin_region']]  # ground truth for eval

stage1_runs = []
stage1_test_preds_by_seed = {}  # seed -> np.array of predicted region ids over test_df
for s in SEEDS:
    r = run_one_with_preds(
        tag=f'stage1_region_seed{s}', seed=s,
        model_checkpoint=ROBERTA_CKPT, learning_rate=ROBERTA_LR, use_class_weights=True,
        train_texts=s1_train_texts, train_labels=s1_train_labels,
        val_texts=s1_val_texts, val_labels=s1_val_labels,
        test_texts_for_preds=s1_test_texts, label_names=region_names,
    )
    test_preds = np.array(r['test_pred_ids'])
    # Compute region metrics against ground truth
    y_true_region = np.array(s1_test_labels)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_region, test_preds, average='macro', zero_division=0)
    bal = balanced_accuracy_score(y_true_region, test_preds)
    acc = accuracy_score(y_true_region, test_preds)
    r.update({
        'test_f1_macro': f1, 'test_bal_acc': bal, 'test_accuracy': acc,
    })
    stage1_runs.append(r)
    stage1_test_preds_by_seed[s] = test_preds

stage1_df = pd.DataFrame([{
    'seed': r['seed'],
    'val_f1_macro':  r['val_f1_macro'],
    'test_f1_macro': r['test_f1_macro'],
    'test_bal_acc':  r['test_bal_acc'],
    'test_accuracy': r['test_accuracy'],
} for r in stage1_runs])
print()
print('Stage 1 region classifier (per seed):')
print(stage1_df.round(4).to_string(index=False))
print()
print('Stage 1 mean ± std:')
print(stage1_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== stage1_region_seed42 | roberta-base | lr=2e-05 | weighted=True | seed=42 ===
{'loss': '2.474', 'grad_norm': '57.79', 'learning_rate': '1.954e-05', 'epoch': '1'}
{'eval_loss': '0.8683', 'eval_accuracy': '0.6717', 'eval_balanced_accuracy': '0.5979', 'eval_precision_macro': '0.6136', 'eval_recall_macro': '0.5979', 'eval_f1_macro': '0.5951', 'eval_runtime': '1.638', 'eval_samples_per_second': '689.9', 'eval_steps_per_second': '21.98', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.475', 'grad_norm': '47.71', 'learning_rate': '1.78e-05', 'epoch': '2'}
{'eval_loss': '0.6954', 'eval_accuracy': '0.7389', 'eval_balanced_accuracy': '0.7043', 'eval_precision_macro': '0.7037', 'eval_recall_macro': '0.7043', 'eval_f1_macro': '0.6946', 'eval_runtime': '1.636', 'eval_samples_per_second': '690.6', 'eval_steps_per_second': '22', 'epoch': '2'}


...

{'loss': '0.2244', 'grad_norm': '43.04', 'learning_rate': '5.373e-06', 'epoch': '9'}
{'eval_loss': '0.9225', 'eval_accuracy': '0.8186', 'eval_balanced_accuracy': '0.7917', 'eval_precision_macro': '0.8053', 'eval_recall_macro': '0.7917', 'eval_f1_macro': '0.7967', 'eval_runtime': '1.689', 'eval_samples_per_second': '669.2', 'eval_steps_per_second': '21.32', 'epoch': '9'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'train_runtime': '316.9', 'train_samples_per_second': '199.7', 'train_steps_per_second': '6.248', 'train_loss': '0.8823', 'epoch': '9'}
{'eval_loss': '0.7142', 'eval_accuracy': '0.8283', 'eval_balanced_accuracy': '0.8076', 'eval_precision_macro': '0.809', 'eval_recall_macro': '0.8076', 'eval_f1_macro': '0.8082', 'eval_runtime': '1.846', 'eval_samples_per_second': '612.1', 'eval_steps_per_second': '19.5', 'epoch': '9'}

Stage 1 region classifier (per seed):
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.8017         0.7969        0.7930         0.8214
  123        0.8040         0.8047        0.7937         0.8285
 2024        0.8082         0.7983        0.7985         0.8223

Stage 1 mean ± std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.8047         0.8000        0.7951         0.8240
std         0.0033         0.0042        0.0030         0.0039


g:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:2458: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## Stage 2 — Per-region country classifiers × 3 seeds each


In [7]:
# For each region: train a RoBERTa country classifier on only that region's rows.
# Predict on the *full* test set (we\'ll route to the right head at eval time).

stage2_runs_by_region = {}  # region -> list of run dicts (one per seed)
# test-set predictions: seed -> region -> np.array of country-name strings length len(test_df)
stage2_test_preds_by_seed_region = {s: {} for s in SEEDS}

for region in region_names:
    tr_mask = (train_df['origin_region'] == region).to_numpy()
    va_mask = (val_df['origin_region']   == region).to_numpy()
    # Countries present in this region's training data
    region_country_names = sorted(train_df.loc[tr_mask, 'origin_country'].unique().tolist())
    region_c2id = {c: i for i, c in enumerate(region_country_names)}
    region_id2c = {i: c for c, i in region_c2id.items()}

    s2_train_texts  = train_df.loc[tr_mask, TEXT_COLUMN].tolist()
    s2_train_labels = [region_c2id[c] for c in train_df.loc[tr_mask, 'origin_country']]
    s2_val_texts    = val_df.loc[va_mask, TEXT_COLUMN].tolist()
    # Val rows whose TRUE country isn\'t in the region\'s training set (rare) get dropped
    val_mask_keep = val_df.loc[va_mask, 'origin_country'].isin(region_c2id).to_numpy()
    s2_val_texts   = [t for t, k in zip(s2_val_texts, val_mask_keep) if k]
    s2_val_labels  = [region_c2id[c] for c, k in zip(val_df.loc[va_mask, 'origin_country'], val_mask_keep) if k]

    runs_this_region = []
    for s in SEEDS:
        tag = f'stage2_{region.replace(" ", "_").replace("-", "_")}_seed{s}'
        r = run_one_with_preds(
            tag=tag, seed=s,
            model_checkpoint=ROBERTA_CKPT, learning_rate=ROBERTA_LR, use_class_weights=True,
            train_texts=s2_train_texts, train_labels=s2_train_labels,
            val_texts=s2_val_texts,     val_labels=s2_val_labels,
            test_texts_for_preds=test_df[TEXT_COLUMN].tolist(),
            label_names=region_country_names,
        )
        pred_country_names = [region_id2c[i] for i in r['test_pred_ids']]
        r['predicted_country_names'] = pred_country_names
        r['region'] = region
        r['region_country_names'] = region_country_names
        runs_this_region.append(r)
        stage2_test_preds_by_seed_region[s][region] = np.array(pred_country_names, dtype=object)

    stage2_runs_by_region[region] = runs_this_region
    print(f'\nStage 2 [{region}] done — region has {len(region_country_names)} countries, '
          f'{tr_mask.sum()} train rows.')


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== stage2_Asia_Pacific_seed42 | roberta-base | lr=2e-05 | weighted=True | seed=42 ===
{'loss': '3.827', 'grad_norm': '3.567', 'learning_rate': '1.961e-05', 'epoch': '1'}
{'eval_loss': '1.938', 'eval_accuracy': '0.06494', 'eval_balanced_accuracy': '0.1429', 'eval_precision_macro': '0.009276', 'eval_recall_macro': '0.1429', 'eval_f1_macro': '0.01742', 'eval_runtime': '0.2316', 'eval_samples_per_second': '665', 'eval_steps_per_second': '21.59', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '3.766', 'grad_norm': '3.559', 'learning_rate': '1.784e-05', 'epoch': '2'}
{'eval_loss': '1.883', 'eval_accuracy': '0.461', 'eval_balanced_accuracy': '0.2803', 'eval_precision_macro': '0.2533', 'eval_recall_macro': '0.2803', 'eval_f1_macro': '0.2292', 'eval_runtime': '0.2293', 'eval_samples_per_second': '671.7', 'eval_steps_per_second': '21.81', 'epoch': '2'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '153.2', 'train_samples_per_second': '79.17', 'train_steps_per_second': '2.506', 'train_loss': '1.859', 'epoch': '11'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '1.103', 'eval_accuracy': '0.6759', 'eval_balanced_accuracy': '0.6199', 'eval_precision_macro': '0.5342', 'eval_recall_macro': '0.6199', 'eval_f1_macro': '0.5589', 'eval_runtime': '0.6132', 'eval_samples_per_second': '352.3', 'eval_steps_per_second': '11.42', 'epoch': '11'}

Stage 2 [South America] done — region has 5 countries, 1011 train rows.


g:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:2458: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## End-to-end evaluation


In [8]:
# Ground truth
y_true_country = test_df['origin_country'].to_numpy()
y_true_region  = test_df['origin_region'].to_numpy()

country_labels_for_macro = sorted(set(y_true_country.tolist()))

eval_rows = []
for s in SEEDS:
    s1_region_ids = stage1_test_preds_by_seed[s]
    s1_region_names = np.array([id2region[i] for i in s1_region_ids])

    # End-to-end country prediction: route each test row to the Stage 2 head indicated by Stage 1
    e2e_country_pred = np.empty(len(test_df), dtype=object)
    for region in region_names:
        mask = (s1_region_names == region)
        if mask.sum() == 0:
            continue
        e2e_country_pred[mask] = stage2_test_preds_by_seed_region[s][region][mask]

    # Oracle-region prediction: use TRUE region to route
    oracle_country_pred = np.empty(len(test_df), dtype=object)
    for region in region_names:
        mask = (y_true_region == region)
        oracle_country_pred[mask] = stage2_test_preds_by_seed_region[s][region][mask]

    # Metrics — macro over all countries present in ground truth
    e2e_f1   = f1_score(y_true_country, e2e_country_pred,    labels=country_labels_for_macro, average='macro', zero_division=0)
    e2e_bal  = balanced_accuracy_score(y_true_country, e2e_country_pred)
    e2e_acc  = accuracy_score(y_true_country, e2e_country_pred)

    ora_f1   = f1_score(y_true_country, oracle_country_pred, labels=country_labels_for_macro, average='macro', zero_division=0)
    ora_bal  = balanced_accuracy_score(y_true_country, oracle_country_pred)
    ora_acc  = accuracy_score(y_true_country, oracle_country_pred)

    # Error decomposition — on the end-to-end errors
    e2e_wrong_mask = (e2e_country_pred != y_true_country)
    wrong_region_mask = e2e_wrong_mask & (s1_region_names != y_true_region)
    wrong_country_mask = e2e_wrong_mask & (s1_region_names == y_true_region)

    total_errors = int(e2e_wrong_mask.sum())
    n_wrong_region = int(wrong_region_mask.sum())
    n_wrong_within = int(wrong_country_mask.sum())

    eval_rows.append({
        'seed': s,
        'stage1_f1':   next(r['test_f1_macro'] for r in stage1_runs if r['seed'] == s),
        'oracle_f1':   ora_f1,
        'oracle_bal':  ora_bal,
        'oracle_acc':  ora_acc,
        'e2e_f1':      e2e_f1,
        'e2e_bal':     e2e_bal,
        'e2e_acc':     e2e_acc,
        'total_errors':  total_errors,
        'wrong_region':  n_wrong_region,
        'wrong_within':  n_wrong_within,
        'frac_wrong_region': (n_wrong_region / total_errors) if total_errors else 0.0,
    })

eval_df = pd.DataFrame(eval_rows)
print('Per-seed summary:')
print(eval_df.round(4).to_string(index=False))
print()
print('Mean ± std across seeds:')
summary_cols = ['stage1_f1', 'oracle_f1', 'oracle_bal', 'oracle_acc', 'e2e_f1', 'e2e_bal', 'e2e_acc', 'frac_wrong_region']
print(eval_df[summary_cols].agg(['mean','std']).round(4).to_string())

print()
print('--- Headline ---')
print(f'Stage 1 region F1:          {eval_df["stage1_f1"].mean():.4f} ± {eval_df["stage1_f1"].std():.4f}')
print(f'Oracle-region country F1:   {eval_df["oracle_f1"].mean():.4f} ± {eval_df["oracle_f1"].std():.4f}')
print(f'End-to-end country F1:      {eval_df["e2e_f1"].mean():.4f} ± {eval_df["e2e_f1"].std():.4f}')
print(f'Cascade penalty:            {(eval_df["oracle_f1"]-eval_df["e2e_f1"]).mean():.4f}')
print(f'Errors from wrong region:   {eval_df["frac_wrong_region"].mean():.2%} of all country errors')
print()
print('Reference — flat 14-class country model (11-compatible split): rerun with country-level floor for direct comparison.')
print('Reference — flat 4-way regional (11): RoBERTa 0.8171 ± 0.0071')
print('Reference — flat 15-class country (07.1): RoBERTa 0.6427 ± 0.0068')


Per-seed summary:
 seed  stage1_f1  oracle_f1  oracle_bal  oracle_acc  e2e_f1  e2e_bal  e2e_acc  total_errors  wrong_region  wrong_within  frac_wrong_region
   42     0.7969     0.5463      0.5775      0.7100  0.4725   0.4861   0.6340           414           202           212             0.4879
  123     0.8047     0.5736      0.5984      0.7569  0.4803   0.4970   0.6658           378           194           184             0.5132
 2024     0.7983     0.5798      0.5904      0.7462  0.4961   0.5082   0.6587           386           201           185             0.5207

Mean ± std across seeds:
      stage1_f1  oracle_f1  oracle_bal  oracle_acc  e2e_f1  e2e_bal  e2e_acc  frac_wrong_region
mean     0.8000     0.5665      0.5888      0.7377  0.4829   0.4971   0.6528             0.5073
std      0.0042     0.0178      0.0105      0.0246  0.0120   0.0111   0.0167             0.0172

--- Headline ---
Stage 1 region F1:          0.8000 ± 0.0042
Oracle-region country F1:   0.5665 ± 0.0178
End-to

## Save results


In [9]:
out = {
    'notebook': '13_Origin_Hierarchical_Region_to_Country',
    'task': 'hierarchical region-then-country classification',
    'text_column': TEXT_COLUMN,
    'country_floor': COUNTRY_FLOOR,
    'n_rows': int(len(work)),
    'n_countries': int(work['origin_country'].nunique()),
    'n_regions': int(work['origin_region'].nunique()),
    'region_mapping': COUNTRY_TO_REGION,
    'region_distribution': work['origin_region'].value_counts().to_dict(),
    'country_distribution': work['origin_country'].value_counts().to_dict(),
    'post_scrub_country_leakage_rate': leak_rate,
    'seeds': SEEDS,
    'stage1_runs': [
        {k: v for k, v in r.items() if k != 'test_pred_ids'} for r in stage1_runs
    ],
    'stage2_runs_by_region': {
        region: [
            {k: v for k, v in r.items() if k not in ('test_pred_ids', 'predicted_country_names', 'label_names')}
            for r in runs
        ]
        for region, runs in stage2_runs_by_region.items()
    },
    'per_seed_eval': eval_rows,
    'summary': {
        'stage1_region_f1_mean':   float(eval_df['stage1_f1'].mean()),
        'stage1_region_f1_std':    float(eval_df['stage1_f1'].std()),
        'oracle_country_f1_mean':  float(eval_df['oracle_f1'].mean()),
        'oracle_country_f1_std':   float(eval_df['oracle_f1'].std()),
        'e2e_country_f1_mean':     float(eval_df['e2e_f1'].mean()),
        'e2e_country_f1_std':      float(eval_df['e2e_f1'].std()),
        'cascade_penalty_mean':    float((eval_df['oracle_f1']-eval_df['e2e_f1']).mean()),
        'frac_wrong_region_mean':  float(eval_df['frac_wrong_region'].mean()),
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/origin_hierarchical_region_to_country_scrubbed_plus\results.json
